<a href="https://colab.research.google.com/github/yutaro-haibara/Kaggle-March-Mania-2026/blob/main/%E2%98%86Predicting_Irrigation_Need_disclose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e4_path = kagglehub.competition_download('playground-series-s6e4')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
path=playground_series_s6e4_path
files = os.listdir(path)
print("今回のコンペでダウンロードされたファイル一覧:")
for f in files:
  print(f"- {f}")


In [ ]:
target_file = "sample_submission.csv"
df = pd.read_csv(os.path.join(playground_series_s6e4_path, target_file))
df.head()

In [ ]:
#データ読み込み
train=pd.read_csv(os.path.join(playground_series_s6e4_path, 'train.csv'))
test_original=pd.read_csv(os.path.join(playground_series_s6e4_path, 'test.csv'))

# test_id を保存
test_id = test_original['id']

# 処理のためにコピーを作成
test = test_original.copy()

train.columns

In [ ]:
train.head()

In [ ]:
# 不要な列
drop_cols = ['id']
train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
!pip install catboost
from catboost import CatBoostClassifier, Pool

In [ ]:
def create_extra_features(df):
    # 1. 水分需給バランス
    #df['Water_Energy_Index'] = (df['Temperature_C'] * df['Sunlight_Hours']) / (df['Humidity'] + 1)

    # 2. 成長段階の重要度（季節との組み合わせ）
    #df['Growth_Context'] = df['Crop_Type'].astype(str) + "_" + df['Crop_Growth_Stage'].astype(str)

    # 3. 風の影響
    #df['Wind_Temp_Interaction'] = df['Wind_Speed_kmh'] * df['Temperature_C']

    #土壌水分を4つのグループに分ける
    #df['Moisture_band']=pd.qcut(df['Soil_Moisture'], q=4, labels=False)


    # 灌漑が必要な可能性を高めるスコア (High score)
    df['high_score'] = 0
    df.loc[df['Soil_Moisture'] < 25, 'high_score'] += 2
    df.loc[df['Rainfall_mm'] < 300, 'high_score'] += 2
    df.loc[df['Temperature_C'] > 30, 'high_score'] += 1
    df.loc[df['Wind_Speed_kmh'] > 10, 'high_score'] += 1

    # 灌漑が不要な可能性を高めるスコア (Low score)
    df['low_score'] = 0
    df.loc[df['Crop_Growth_Stage'] == 'Harvest', 'low_score'] += 2
    df.loc[df['Crop_Growth_Stage'] == 'Sowing', 'low_score'] += 2
    df.loc[df['Mulching_Used'] == 'Yes', 'low_score'] += 1

    # 総合スコア
    df['formula_score'] = df['high_score'] - df['low_score']

    # スコアに基づく予測ラベル（これをモデルに直接教える）
    df['formula_pred'] = 'Medium'
    df.loc[df['formula_score'] > 0, 'formula_pred'] = 'High'
    df.loc[df['formula_score'] < 0, 'formula_pred'] = 'Low'

    # 作物による補正 (Rice, Sugarcane, Bananaなどは水需要が高い)

    thirsty_crops = ['Rice', 'Sugarcane', 'Banana']
    df.loc[df['Crop_Type'].isin(thirsty_crops), 'high_score'] += 1

    # 成長段階による補正 (Vegetative: 栄養成長期は水が必要)
    df.loc[df['Crop_Growth_Stage'] == 'Vegetative', 'high_score'] += 0.5

    # これにより Low になりすぎるのを防いでMediumを多くする
    df.loc[df['Mulching_Used'] == 'Yes', 'low_score'] -= 0.5

    # 雨が多くても(>1000)、土が乾いている(<20)or暑すぎる(>35)なら加点
    df.loc[(df['Rainfall_mm'] > 1000) & ((df['Soil_Moisture'] < 20) | (df['Temperature_C'] > 35)), 'high_score'] += 1.0

    # 3. 特定の地域（Region）の乾燥補正
    # エラーデータに多い地域（例：WestやNorthなど）があれば微調整
    df.loc[df['Region'] == 'West', 'high_score'] += 0.5

    # Vegetative
    df.loc[df['Crop_Growth_Stage'] == 'Vegetative', 'high_score'] += 1.5

    # Flowering
    df.loc[df['Crop_Growth_Stage'] == 'Flowering', 'high_score'] += 1.2


    # 低湿度の時の Flowering はさらに危険なので加点
    df.loc[(df['Crop_Growth_Stage'] == 'Flowering') & (df['Humidity'] < 40), 'high_score'] += 0.5

    #スコアの最終確定
    df['formula_score'] = df['high_score'] - df['low_score']

    #判定ラベルの出力
    df['formula_pred'] = 'Medium'
    df.loc[df['formula_score'] > 0.5, 'formula_pred'] = 'High'
    df.loc[df['formula_score'] < -1, 'formula_pred'] = 'Low'
    return df

train = create_extra_features(train)
test = create_extra_features(test)

cat_features = train.select_dtypes(include=['object']).columns.tolist()
print(f"Category Features: {cat_features}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def add_mathematical_features(train_df, test_df):
    # 1. 数値データの抽出と標準化
    # センサーデータなど、数値の列だけを指定
    num_cols = ['Soil_Moisture', 'Temperature_C', 'Humidity', 'Wind_Speed_kmh', 'Sunlight_Hours']

    scaler = StandardScaler()
    train_num_scaled = scaler.fit_transform(train_df[num_cols])
    test_num_scaled = scaler.transform(test_df[num_cols])

    #K-means
    n_clusters = 5
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)

    train_df['Cluster_ID'] = kmeans.fit_predict(train_num_scaled).astype(str) # カテゴリとして扱う
    test_df['Cluster_ID'] = kmeans.predict(test_num_scaled).astype(str)

    #PCA (主成分分析)
    # 5つの数値を、情報の密度が高い「2つの合成変数」に凝縮する
    pca = PCA(n_components=2, random_state=42)
    train_pca = pca.fit_transform(train_num_scaled)
    test_pca = pca.transform(test_num_scaled)

    train_df['PCA_1'] = train_pca[:, 0] # 第1主成分
    train_df['PCA_2'] = train_pca[:, 1] # 第2主成分
    test_df['PCA_1'] = test_pca[:, 0]
    test_df['PCA_2'] = test_pca[:, 1]

    return train_df, test_df

# 実行
train, test = add_mathematical_features(train, test)

if 'Cluster_ID' not in cat_features:
    cat_features.append('Cluster_ID')

In [ ]:
print(train['Irrigation_Need'].value_counts(dropna=False))

In [ ]:
features = [col for col in train.columns if col not in ['id', 'Irrigation_Need']]
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
# X と X_test を同じ列構成で作る
X = train[features]
y = train['Irrigation_Need'].map(target_map)

X_test = test[features]

In [ ]:
# 文字列型の列名を取得
cat_features = X.select_dtypes(include=['object']).columns.tolist()
print(f"Category Features: {cat_features}")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
!pip install optuna
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

In [ ]:
def objective(trial):
    # 探索するパラメータの範囲
    params = {
    'learning_rate': 0.07966478890229094,
    'depth': 5,
    'l2_leaf_reg': 4.585052325393321,
    'random_strength': 3.7299239940542472,
    'bagging_temperature': 0.5659468003635836,
    'loss_function': 'MultiClass',
    'eval_metric': 'Accuracy',
    'iterations': 2000,
    'random_seed': 42,
    'task_type': 'GPU'  # GPUを使う場合
}
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), early_stopping_rounds=50)

    # 検証データでの精度を評価
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

In [ ]:
## 最適化のセッションを作成
#study = optuna.create_study(direction="maximize")
# n_trialsはまずは20〜50回程度で試してみるのがオススメです
#study.optimize(objective, n_trials=30)

#print("Best Score:", study.best_value)
#print("Best Params:", study.best_params)

In [ ]:
import lightgbm as lgb

# カテゴリ変数を 'category' 型に変換
for col in cat_features:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 再度分割（型変換を反映させるため）
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 各特徴量に対する重みをリストで作成する

weights_dict = {
    'formula_score': 5,
    'formula_pred': 5,
    'high_score': 2,
    'low_score': 2
}
# X.columns の各列に対して重みを割り当て
feature_weights = [weights_dict.get(col, 1.0) for col in X.columns]
# LGBMのパラメータ（これまでの傾向に合わせた推奨値）
lgbm_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'max_depth': 6,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'feature_weight': feature_weights,
    'random_state': 42,
    'n_estimators': 2000,
    'device': 'cpu' # GPUを使う場合
}

# モデルの定義
lgbm_model = lgb.LGBMClassifier(**lgbm_params)

# 学習
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)]
)

In [ ]:

# 1. パラメータを定義
my_params = {
    'learning_rate': 0.07966478890229094,
    'depth': 5,
    'l2_leaf_reg': 4.585052325393321,
    'random_strength': 3.7299239940542472,
    'bagging_temperature': 0.5659468003635836,
    'loss_function': 'MultiClass',
    'eval_metric': 'Accuracy',
    'iterations': 2000,
    'random_seed': 42,
    'task_type': 'CPU'  # GPUを使う場合
}

# 特徴量の名前に合わせて重みを設定
weights = {
    'formula_score': 5.0,  # 5倍重視
    'formula_pred': 5.0,   # 5倍重視
    'high_score': 2.0,
    'low_score': 2.0
}

# 全ての特徴量リストに対して重みを割り当て（指定がないものは1.0）
feature_weights = [weights.get(col, 1.0) for col in X.columns]

model = CatBoostClassifier(
    **my_params,
    feature_weights=feature_weights, # ここで重みを指定
    verbose=100
)

cat_features_for_fit = [f for f in cat_features if f in X_train.columns]

# 学習
model.fit(
    X_train, y_train,
    cat_features=cat_features_for_fit,
    eval_set=(X_val, y_val),    # 検証データ
    use_best_model=True,
    early_stopping_rounds=50
)

In [ ]:
# バリデーション評価
preds = model.predict(X_val)
print(classification_report(y_val, preds))

# テストデータでの予測
test_preds = model.predict(X_test)

# 提出用ファイルの作成
submission = pd.read_csv(os.path.join(playground_series_s6e4_path, 'sample_submission.csv'))
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}
submission['Irrigation_Need'] = [reverse_map[p[0]] for p in test_preds]
submission.to_csv('baseline_submission.csv', index=False)

In [ ]:
# 各モデルで「確率（predict_proba）」を出す

# CatBoostの予測結果を評価
# 確率からクラスラベルに変換して評価
# print(classification_report(y_val, preds))
# LGBMの予測結果
lgbm_val_probs = lgbm_model.predict_proba(X_val) # バリデーションデータで確率を出す
lgbm_val_preds = lgbm_val_probs.argmax(axis=1) # 確率から予測クラスに変換
print("LGBM Validation Report:")
print(classification_report(y_val, lgbm_val_preds))

# 5:5 でブレンド
# CatBoostの予測確率（eval_setで使われたX_valに対するもの）
catboost_val_probs = model.predict_proba(X_val)

# アンサンブルの予測確率（バリデーションセット）
final_val_probs = (catboost_val_probs + lgbm_val_probs) / 2
final_val_preds = final_val_probs.argmax(axis=1)
print("Ensemble Validation Report:")
print(classification_report(y_val, final_val_preds))

# テストデータでの予測
# LGBMのテスト予測確率
lgbm_test_probs = lgbm_model.predict_proba(X_test)
# LGBMのテスト予測クラス
lgbm_test_preds = lgbm_test_probs.argmax(axis=1)

# アンサンブルのテスト予測（CatBoostのテスト予測確率とLGBMのテスト予測確率をブレンド）
final_test_probs = (model.predict_proba(X_test) + lgbm_test_probs) / 2
final_test_preds = final_test_probs.argmax(axis=1)

# 提出用ファイルの作成 (LGBM単体)
submission = pd.read_csv(os.path.join(playground_series_s6e4_path, 'sample_submission.csv'))
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}

# LGBMのテストセットに対する予測結果を代入
submission['Irrigation_Need'] = [reverse_map[p] for p in lgbm_test_preds]

submission.to_csv('lgbm_submission.csv', index=False) # ファイル名を変更

# CatBoostのテスト予測で提出
submission_catboost = pd.read_csv(os.path.join(playground_series_s6e4_path, 'sample_submission.csv'))
submission_catboost['Irrigation_Need'] = [reverse_map[p[0]] for p in model.predict(X_test)]
submission_catboost.to_csv('catboost_submission.csv', index=False)

# アンサンブルで提出する
submission_ensemble = pd.read_csv(os.path.join(playground_series_s6e4_path, 'sample_submission.csv'))
submission_ensemble['Irrigation_Need'] = [reverse_map[p] for p in final_test_preds]
submission_ensemble.to_csv('ensemble_submission.csv', index=False)

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.metrics import accuracy_score

# def objective(trial):
#     param = {
#         'objective': 'multiclass',
#         'num_class': 3,
#         'metric': 'multi_logloss',
#         'verbosity': -1,
#         'boosting_type': 'gbdt',
#         'random_state': 42,
#         'device': 'cpu', # GPU使用時

#         # 最適化したい範囲を指定
#         'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
#         'num_leaves': trial.suggest_int('num_leaves', 31, 256),
#         'max_depth': trial.suggest_int('max_depth', 5, 15),
#         'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
#         'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
#         'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
#         'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),

#         # あなたが設定した feature_weight もここで渡す
#         'feature_weight': feature_weights
#     }

#     # 学習（簡易的な分割例）
#     model = lgb.LGBMClassifier(**param)
#     model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=50)])

#     preds = model.predict(X_val)
#     accuracy = accuracy_score(y_val, preds)
#     return accuracy

# # 最適化の実行
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=50) # 50回ほど試行

# print('Best trial:', study.best_trial.params)

#スコアは上がらなかった

In [ ]:
# 特徴量の重要度を表示
import matplotlib.pyplot as plt

feature_importance = model.get_feature_importance()
feature_names = X.columns
sorted_idx = feature_importance.argsort()

plt.figure(figsize=(10, 8))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
plt.title('Feature Importance')
plt.show()

In [ ]:
# 検証データで間違えたものを特定
val_preds = model.predict(X_val)
# 1次元配列に変換（CatBoostの戻り値対策）
val_preds = val_preds.flatten()

error_df = X_val.copy()
error_df['actual'] = y_val.values
error_df['pred'] = val_preds

errors = error_df[error_df['actual'] != error_df['pred']]

print(f"間違いの数: {len(errors)}")
display(errors.head(10))

In [ ]:
errors.to_csv('model_errors.csv', index=False)
print("'model_errors.csv' としてエクスポートしました")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import pandas as pd

# エラーデータの読み込み
errors = pd.read_csv('model_errors.csv')

# 行列の作成
cm = confusion_matrix(errors['actual'], errors['pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Low(0)', 'Medium(1)', 'High(2)'],
            yticklabels=['Low(0)', 'Medium(1)', 'High(2)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Error Patterns (Where the model failed)')
plt.show()

In [ ]:
# 分析したいカテゴリ変数のリスト
cat_cols = ['Crop_Type', 'Crop_Growth_Stage', 'Region', 'Mulching_Used']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # エラーデータ内での各カテゴリの出現頻度
    error_counts = errors[col].value_counts()
    sns.barplot(x=error_counts.index, y=error_counts.values, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Errors by {col}')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 分析したいカテゴリ変数のリスト
cat_cols = ['Crop_Type', 'Crop_Growth_Stage', 'Region', 'Mulching_Used']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # 元の全データ(trainまたはX_val)での各カテゴリの件数
    total_counts = train[col].value_counts()

    # エラーデータ内での各カテゴリの件数
    error_counts = errors[col].value_counts()

    # エラー率（割合）を計算 (エラー数 / 全データ数)
    # reindexを使用して計算
    error_rate = (error_counts / total_counts).reindex(total_counts.index).fillna(0) * 100

    # 可視化 (エラー率が高い順にソート)
    error_rate = error_rate.sort_values(ascending=False)
    sns.barplot(x=error_rate.index, y=error_rate.values, ax=axes[i], palette='magma')

    axes[i].set_title(f'Error Rate by {col} (%)')
    axes[i].set_ylabel('Error Rate (%)')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     73983
           1       0.98      0.98      0.98     47815
           2       0.96      0.91      0.94      4202

    accuracy                           0.98    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000